## Step 2: Load and Inspect

In [ ]:
import pandas as pd
df = pd.read_csv("../data/raw/netflix_titles.csv")

In [ ]:
df.shape

In [ ]:
df.head()

Dataset contains 8,807 rows and 12 columns, matching the 12 expected fields (show_id, type, title, director, cast, country, date_added, release_year, rating, duration, listed_in, description).

In [ ]:
df.info()

**Missing values (from .info()):**
- director: 6,173 non-null out of 8,807 -> 2,634 missing (~30%)
- cast: 7,982 non-null -> 825 missing (~9%)
- country: 7,976 non-null -> 821 missing (~9%)
- date_added, racing, duration: negligible missingness (<15 rows each)

**Data types:** data_added and duration are both stored as 'str', not as real date or numeric type, despite looking like a date/number

In [ ]:
(df.isnull().sum() / df.shape[0] * 100).sort_values(ascending=False)

**Missing value % (sorted):**
- director: 29.9%
- country: 9.44%
- cast: 9.37%
- all other columns: <1%

In [ ]:
df[df['type'] == 'Movie']['duration'].head(3)

In [ ]:
df[df['type'] == 'TV Show']['duration'].head(3)

**Structural findings:**
- 'type' has exactly two clean categories: Movie, TV Show.
- 'duration' means different things per type: "90 min" for Movies vs.
  "2 Seasons"/"1 Season" for TV Shows - will need to be split into a 
  numeric value + unit, conditional on type.
- 'listed_in' holds multiple comma-separated genres per row (e.g.
  "International TV Shows, TV Dramas, TV Mysteries") - will need to be 
  split into individual genres for genre-level analysis.

## Step 3: Cleaning

Based on the issues identified in Step 2, this section addresses four problems in order: missing values in director/cast/country, converting date_added to a real dat, splitting duration into a usable numeric value, and handling the multi-genre listed_in column.

In [ ]:
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')

In [ ]:
df.isnull().sum()

**Handling missing values in director, cast, country:**
Filled missing values with "Unknown" rather than dropping rows, since these columns are non-numeric and other analysis questions (e.g. titles per year, genre trends) don't depend on director/cast/country being present. Dropping ~30 of rows (the missingness in 'director' alone) would unnecessarily shrink the dataset for unrelated analyses.

In [ ]:
df['date_added'] = df['date_added'].str.strip()
df['date_added'] = pd.to_datetime(df['date_added'])

In [ ]:
df['date_added'].head()

In [ ]:
df.info()

**Converting date_added to a real date:**
Some values had inconsistent leading whitespace (e.g. " August 4, 2027"), which broke pandas' single-format date parsing. Stripped whitespace first, then converted to datetime64 using pd.to_datetime(). The 10 originally-missing values remain as NaT (negligble, ~0.1% of rows) - left as-is rather than filled, since there's no sensible date to impute here.

In [ ]:
df['duration'].str.split(' ')

In [ ]:
df[df['duration'].isnull()]

In [ ]:
df.loc[5541]

In [ ]:
shifted_rows = df['duration'].isnull()

In [ ]:
df.loc[shifted_rows, 'duration'] = df.loc[shifted_rows, 'rating']

In [ ]:
df.loc[shifted_rows, 'rating'] = 'Unknown'

In [ ]:
df.loc[5541]

In [ ]:
df[df['rating'].isnull()]

In [ ]:
df['rating'].isnull().sum()

**Fixing shifted data (rating/duration/listed_in):**
Found 3 rows where source data had shifted one column: the real duration value ("74 min" etc.) was sitting in 'rating', and 'listed_in' was truncated to just "Movies". Recovered the duration value into the correct column, and set 'rating' to "Unknown" for these rows since the true rating was lost upstream. Confirmed no other missing ratings remain.

In [ ]:
duration_split = df['duration'].str.split(' ')

In [ ]:
df['duration_value'] = duration_split.str[0]

In [ ]:
df['duration_value'].head()

In [ ]:
df['duration_value'] = pd.to_numeric(df['duration_value'])

In [ ]:
df[['duration', 'duration_value']].head(10)

In [ ]:
df['duration_value'].dtype

In [ ]:
df['duration_unit'] = duration_split.str[1]

In [ ]:
df['duration_unit'].unique()

In [ ]:
df[['type', 'duration', 'duration_value', 'duration_unit']].head(10)

**Splitting 'duration' into 'duration_value' and 'duration_unit'::**
The original 'duration' column mixed two different units depending on 'type' ("90 min" for Movies, "2 Seasons"/"1 Season" for TV Shows), making it unusable for numeric analysis as-is. Split on the space character into a numeric 'duration_value' (int64, confirmed zero missing after fixing the shifted-column bug above) and a text 'duration_unit' (min/Season/Seasons).

Kept the original 'duration' column alongside the split columns, rather than dropping it, to preserve the raw soruce value for transparency - this lefts anyone reviewing the cleaning logic directly compare the parsed output against the original text without needing to re-derive it. 

In [ ]:
df['genre_list'] = df['listed_in'].str.split(', ')

In [ ]:
df['genre_list'].head()

In [ ]:
df['listed_in'].isnull().sum()

In [ ]:
df_by_genre = df.explode('genre_list')

In [ ]:
df['genre_list'].head()

In [ ]:
df['genre_list'].isnull().sum()

In [ ]:
df_by_genre[['title', 'genre_list']].head(10)

In [ ]:
df.shape

In [ ]:
df_by_genre.shape

**Handling listed_in (multi-genre column):**
Split the comma-separated 'listed_in' string into a list column ('genre_list') using str.split(', '). Since genre-level analysis (top genres, genre trends) needs genre to behave as an individual category rather than a combined string, exploded genre_list into a separate DataFrame ('df_by_genre') with one row per title-genre pair, rather than overwriting the original 'df'. This preserves the one-row-per-title version for other analysis, while making df_by_genre available for genre-specific aggregation (e.g. count of titles per genre).

## Step 4: Explore with Pandas

Now that the data is cleaned, this section digs into a few concrete questions using pandas:

1. Has the number of titles added to Netflix grown year over year, and is that growth different for Movies vs. TV Shows?
2. What are the top 10 most common genres, and has that pattern been consistent over the years?
3. What's the average movie length, and is there a tendency toward longer, shorter, or middle-length movies?
4. Which countries contribute the most content to Netflix? (Note: "Unknown" is included as its own category here, since ~9% of country values were missing and dilled with "Unknown" in Step 3 - it's called out explicitly rather than silently dropped, so the missing-data context isn't lost.)

In [ ]:
df['year_added'] = df['date_added'].dt.year

In [ ]:
df['year_added'].value_counts()

In [ ]:
df['year_added'].dropna().value_counts().sort_index()

In [ ]:
df.groupby(['year_added', 'type']).size()

In [ ]:
year_type_counts = df.groupby(['year_added', 'type']).size().unstack()
year_type_counts

In [ ]:
year_type_counts.index = year_type_counts.index.astype(int)
year_type_counts

**Q1: Growth in titles added over time (Movies vs. TV Shows)**

Titles added to Netflix grew substantially from 2008 through 2019, with the steepest growth between 2016-2019 (429 -> 1188 -> 1649 -> 2016 titles/year). Both 2020 and 2021 show a decline from the 2019 peak (1879, then 1498) - plausibly related to production slowdowns during COVID-19, though this dataset alone can't confirm causation.

Movies have outpaced TV Shows in raw count every single year, though TV Shows grew at a comparable rate - e.g. in 2019, 1424 Movies vs. 592 TV Shows were added, roughly a 2.4x ratio, fairly consistent across the later years.

Note: ~10 rows with missing 'date_added' are excluded from this year-by-year breakdown, since groupby drops rows with a missing group key by default.

In [ ]:
df_by_genre.columns

In [ ]:
df_by_genre = df.explode('genre_list')
df_by_genre.columns

In [ ]:
top_genres = df_by_genre['genre_list'].value_counts()[:10]
top_genres

In [ ]:
top_genre_names = top_genres.index
df_top_genres = df_by_genre[df_by_genre['genre_list'].isin(top_genre_names)]

In [ ]:
df_top_genres.shape

In [ ]:
genre_trend = df_top_genres.groupby(['year_added', 'genre_list']).size().unstack()
genre_trend.index = genre_trend.index.astype(int)
genre_trend = genre_trend.fillna(0).astype(int)
genre_trend

**Q2: Top genres and consistency over time**

Top 10 genres overall: International Movies (2752), Dramas (2427), Comedies (1674), International TV Shows (1351), Documentaries (869), Action & Adventure (859), TV Dramas (763), Independent Movies (756), Children & Family Movies (641), Romantic Movies (616).

Though International Movies and Dramas show consistently high counts from 2016 onward, they don't peak in the same year: International Movies peaks in 2018 (668) and already begins declining in 2019 (610), a year ahead of Dramas, which peaks in 2019 (564) before declining. Most other top genres follow the Dramas pattern - growing through 2019, then declining in 2020-2021. Romantic Movies is a notable exception in the other direction, continuing to grow through 2020 (173) before dropping in 2021.

In [ ]:
movies = df[df['type'] == 'Movie']

In [ ]:
movies['duration_value'].mean()

In [ ]:
movies['duration_unit'].unique()

In [ ]:
movies['duration_value'].describe()

In [ ]:
movies[movies['duration_value'] == movies['duration_value'].min()][['title', 'duration_value', 'listed_in']]

In [ ]:
movies[movies['duration_value'] == movies['duration_value'].max()][['title', 'duration_value', 'listed_in']]

**Q3: Average movie length and duration distribution**

Movies average around 99 minutes (median: 98 minutes), and there's a clear tendency toward middle-length films: the middle 50% of all movies fall between 87 and 114 minutes, a fairly tight cluster around standard feature-film runtime. The closeness of the mean and median suggests this figure is representative of the dataset as a whole, rather than skewed by outlets – even though a few extreme cases exist at the edges, like 'Black Mirror: Bandersnatch' (312 minutes, an interactive film whose length reflect its branching structure) and 'Silent' (3 minutes) - no clear explanation for this runtime is available from the dataset's other columns, though it may be a short film. 

In [ ]:
df['country'].head(20)

In [ ]:
df['country_list'] = df['country'].str.split(', ')
df_by_country = df.explode('country_list')

In [ ]:
top_countries = df_by_country['country_list'].value_counts()[:10]
top_countries

In [ ]:
top_countries_known = df_by_country[df_by_country['country_list'] != 'Unknown']['country_list'].value_counts()[:10]
top_countries_known

**Q4: Country contribution on Netflix:**

The United States leads by a wide margin (3689 country mentions), roughly 3.5x India's total (1046) - the two clear leaders, with a sharp drop-off after that. The rest of the top 10: United Kingdom (804), Canada (445), France (393), Japan (318), Spain (232), South Korea (231), Germany (226), and Mexico (169). 

Note: "Unknown" (831 entries, ~9% of country data) was excluded from this ranked list to avoid mixing missing data in with real countries - but it's worth noting it would rank 3rd overall, ahead of the UK, if include. 

Also note: these are country mentions, not unique titles - a co-produced title (e.g. one listing "United States, Ghana, Burkina Faso, United Kingdom") is counted once for each country listed, so these numbers reflect how many titles each country contributed to in some capacity, not titles produced exclusively by that country. 

## Step 5: Same Questions in SQL

Loaded the cleaned data into a local SQLite database (data/processed/netflix.db) using three tables: 'titles' (one row per title, for Q1/Q3), 'titles_by_genre' (one row per title-genre pair, for Q2), and 'titles_by_country' (one row per title-country pair, for Q4). List-type columns (genre_list, country_list) were excluded from tables that didn't need them, since SQLite can't store Python list objects directly.

In [ ]:
import sqlite3
import os

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
conn = sqlite3.connect('../data/processed/netflix.db')

In [ ]:
df_sql = df.drop(columns=['genre_list', 'country_list'])
df_by_genre = df.explode('genre_list')
df_by_country = df.explode('country_list')

In [ ]:
df_sql.to_sql('titles', conn, if_exists='replace', index=False)
df_by_genre.drop(columns=['country_list']).to_sql('titles_by_genre', conn, if_exists='replace', index=False)
df_by_country.drop(columns=['genre_list']).to_sql('titles_by_country', conn, if_exists='replace', index=False)


In [ ]:
print(pd.read_sql_query("SELECT COUNT(*) FROM titles", conn))
print(pd.read_sql_query("SELECT COUNT(*) FROM titles_by_genre", conn))
print(pd.read_sql_query("SELECT COUNT(*) FROM titles_by_country", conn))

In [ ]:
q1_sql = pd.read_sql_query("""
    SELECT
        strftime('%Y', date_added) AS year_added,
        type,
        COUNT(*) AS title_count
    FROM titles
    WHERE date_added IS NOT NULL
    GROUP BY year_added, type
    ORDER BY year_added;
""", conn)
q1_sql

**Q1 (SQL): Titles added per year, by type**

Rewrote the pandas year-by-type breakdown as a SQL query using strftime() to extract the year from date_added, GROUP BY year and type, with COUNT(*) for the title count. Results match the pandas analysis exactly (e.g. 2019: 1424 Movies / 592 TV Shows; 2008: 1 / 1), confirming the translation is correct.

In [ ]:
q2_sql = pd.read_sql_query("""
    SELECT
        genre_list AS genre,
        COUNT(*) AS title_count
    FROM titles_by_genre
    GROUP BY genre_list
    ORDER BY title_count DESC
    LIMIT 10;
""", conn)
q2_sql

**Q2 (SQL): Top 10 genres**

Rewrote the pandas genre ranking as a SQL query against titles_by_genre, using GROUP BY genre_list with COUNT(*), ORDER BY DESC, and LIMIT 10. Results match the pandas analysis exactly, confirming the translation (International Movies: 2752, Dramas: 2427, down through Romantic Movies: 616).

In [ ]:
q3_sql = pd.read_sql_query("""
    SELECT
        AVG(duration_value) AS avg_duration,
        MIN(duration_value) AS min_duration,
        MAX(duration_value) AS max_duration
    FROM titles
    WHERE type = 'Movie';
""", conn)
q3_sql

In [ ]:
q3_sql_percentiles = pd.read_sql_query("""
    WITH ranked AS (
        SELECT 
            duration_value,
            NTILE(4) OVER (ORDER BY duration_value) AS quartile
        FROM titles
        WHERE type = 'Movie'
    )
    SELECT 
        quartile,
        MIN(duration_value) AS min_val,
        MAX(duration_value) AS max_val,
        COUNT(*) AS n
    FROM ranked
    GROUP BY quartile;
""", conn)
q3_sql_percentiles

**Q3 (SQL): Average movie length and duration distribution**

Rewrote the pandas duration analysis in two parts. First, AVG/MIN/MAX against titles filtered to type = 'Movie' matched pandas exactly (avg: 99.56, min: 3, max: 312). Second, since SQLite has no built-in percentile function, used a window function (NTILE(4) OVER (ORDER BY duration_value)) inside a CTE to splot movies into quartiles, then took MIN/MAX per quartile as approximate percentile boundaries. The results matched pandas' 25th/50th/75th percentiles exactly: 87 / 98 / 114 minutes. 

In [ ]:
q4_sql = pd.read_sql_query("""
    SELECT
        country_list AS country,
        COUNT(*) AS title_count
    FROM titles_by_country
    WHERE country_list != 'Unknown'
    GROUP BY country_list
    ORDER BY title_count DESC
    LIMIT 10;
""", conn)
q4_sql

**Q4 (SQL): Top 10 countries**

Rewrote the pandas country ranking as a SQL query against titles_by_country (not titles, since the raw country column still holds comma-separated co-prodution strings that would undercount individual countries if queried directly). Used WHERE country_list != 'Unknown' to exclude the placeholder, GROUP BY/COUNT/ORDER BY DESC/LIMIT 10 for the ranking. Results match pandas exactly, from United States (3689) down to Mexico (169).

## Step 6: Export

Exported four CSVs to outputs/, ready for Tableau
- netflix_titles_cleaned.csv - full cleaned dataset (covers Q3 directly; Tableau can aggregate duration_value on its own for average/distribution)
- titles_by_year_and_type.csv - Q1 summary (titles per year, Movie vs TV Show)
- genre_trend_by_year.csv - Q2 Summary (top genres per year)
- top_countries.csv - Q4 summary (top 10 countries by content count)

In [ ]:
df.to_csv('../outputs/netflix_titles_cleaned.csv', index=False)
year_type_counts.to_csv('../outputs/titles_by_year_and_type.csv', index=False)
genre_trend.reset_index().to_csv('../outputs/genre_trend_by_year.csv', index=False)
top_countries_known.reset_index().to_csv('../outputs/top_countries.csv', index=False)